# WESAD HRV Stress Detection - Extended Analysis (v2)

Adds, on top of the main study:
1. **Reproducibility**: pinned versions, a per-window manifest (with rejection reasons),
   saved per-subject predictions, and this one notebook regenerates everything.
2. **Variability-without-mean-HR ablation** + a majority-class baseline, so we test whether
   variability predicts stress *independently* of heart rate.
3. **Signal-validity check**: compare PPG-derived HRV against WESAD's synchronized **ECG**,
   and split windows by **motion** (wrist accelerometer) to see if artifacts drive results.

Run: open in Colab -> Runtime -> Run all. ~20-30 min (ECG peak detection adds time).


In [ ]:
# 1. Dependencies (pin neurokit2; others are Colab defaults, logged below for reproducibility)
!pip -q install neurokit2==0.2.10

In [ ]:
# 2. Download + unzip WESAD (~2 GB, in Colab)
import os, zipfile
DATA_DIR="/content/wesad"; ZIP_PATH="/content/WESAD.zip"
WESAD_URL="https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"  # replace if it fails
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATA_DIR,"WESAD","S2","S2.pkl")):
    if not os.path.exists(ZIP_PATH):
        print("Downloading WESAD (~2 GB)..."); import subprocess
        subprocess.run(["wget","-q","-O",ZIP_PATH,WESAD_URL])
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DATA_DIR)
print("Subjects:", sorted(os.listdir(os.path.join(DATA_DIR,"WESAD"))))

In [ ]:
# 3. Imports + configuration (+ version log for reproducibility)
import pickle, warnings, platform, sys
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import neurokit2 as nk, sklearn, scipy
from scipy import signal as spsig, interpolate as spinterp, stats as spstats
from scipy.integrate import trapezoid
from tqdm import tqdm

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__, "| scipy", scipy.__version__, "| neurokit2", nk.__version__)

WESAD_ROOT=os.path.join(DATA_DIR,"WESAD")
SUBJECTS=[f"S{i}" for i in [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]]
BVP_FS=64; ECG_FS=700; ACC_FS=32; LABEL_FS=700
WINDOW_SEC=60; STRIDE_SEC=30
LABEL_MAP={1:0, 2:1}          # baseline->0, stress->1
PURITY=0.90; MIN_BEATS=12
OUT="/content/outputs_v2"; os.makedirs(OUT, exist_ok=True)
PPG_FEATS=["meanHR","SDNN","RMSSD","pNN50","LF","HF","LFHF"]
VARIABILITY_FEATS=["SDNN","RMSSD","pNN50","LF","HF","LFHF"]   # excludes meanHR
TIME_FEATS=["meanHR","SDNN","RMSSD","pNN50"]; FREQ_FEATS=["LF","HF","LFHF"]

In [ ]:
# 4. HRV helpers (time + frequency) shared by PPG and ECG
def freq_domain(rri):
    t=np.cumsum(rri)/1000.0; t=t-t[0]
    if t[-1]<20 or len(rri)<8: return np.nan,np.nan,np.nan
    ti=np.arange(0,t[-1],0.25)
    x=spinterp.interp1d(t,rri,kind="cubic",fill_value="extrapolate")(ti); x=x-x.mean()
    f,p=spsig.welch(x,fs=4.0,nperseg=min(len(x),256))
    lf=trapezoid(p[(f>=0.04)&(f<0.15)], f[(f>=0.04)&(f<0.15)])
    hf=trapezoid(p[(f>=0.15)&(f<0.40)], f[(f>=0.15)&(f<0.40)])
    return lf,hf,(lf/hf if hf>0 else np.nan)

def hrv_from_rri(rri):
    rri=rri[(rri>=300)&(rri<=2000)]
    if len(rri)<MIN_BEATS-1: return None
    d=np.diff(rri); lf,hf,lfhf=freq_domain(rri)
    return dict(meanHR=60000.0/np.mean(rri), SDNN=np.std(rri,ddof=1),
                RMSSD=np.sqrt(np.mean(d**2)), pNN50=np.mean(np.abs(d)>50)*100,
                LF=lf, HF=hf, LFHF=lfhf)

def ppg_rri(bvp):
    try:
        c=nk.ppg_clean(bvp, sampling_rate=BVP_FS)
        pk=nk.ppg_findpeaks(c, sampling_rate=BVP_FS)["PPG_Peaks"]
    except Exception: return None
    if len(pk)<MIN_BEATS: return None
    return np.diff(pk)/BVP_FS*1000.0

def ecg_rri(ecg):
    # fast band-pass + peak-find R-peak detector (much quicker than full neurokit on 700 Hz)
    try:
        b,a=spsig.butter(3,[5/(ECG_FS/2),20/(ECG_FS/2)],btype="band")
        fe=spsig.filtfilt(b,a,ecg)
        pk,_=spsig.find_peaks(fe, distance=int(0.4*ECG_FS), height=np.std(fe)*1.5)
    except Exception: return None
    if len(pk)<MIN_BEATS: return None
    return np.diff(pk)/ECG_FS*1000.0

In [ ]:
# 5. Window every subject: PPG-HRV + ECG-HRV + motion + a manifest with rejection reasons
rows=[]; manifest=[]
for subj in tqdm(SUBJECTS, desc="subjects"):
    with open(os.path.join(WESAD_ROOT,subj,f"{subj}.pkl"),"rb") as fh:
        data=pickle.load(fh, encoding="latin1")
    bvp=np.asarray(data["signal"]["wrist"]["BVP"]).flatten()
    ecg=np.asarray(data["signal"]["chest"]["ECG"]).flatten()
    wacc=np.asarray(data["signal"]["wrist"]["ACC"]).astype(float)   # (N,3) @32Hz
    label=np.asarray(data["label"]).flatten()
    wlen, stride = WINDOW_SEC*BVP_FS, STRIDE_SEC*BVP_FS
    for i in range(0, len(bvp)-wlen, stride):
        t0,t1=i/BVP_FS,(i+wlen)/BVP_FS
        lab=label[int(t0*LABEL_FS):int(t1*LABEL_FS)]
        rec=dict(subject=subj,t0=round(t0,2),t1=round(t1,2))
        if len(lab)==0: rec.update(label=None,kept=False,reason="no_label"); manifest.append(rec); continue
        v,c=np.unique(lab,return_counts=True); dom=int(v[np.argmax(c)]); pur=c.max()/c.sum()
        if dom not in LABEL_MAP: rec.update(label=dom,kept=False,reason="other_condition"); manifest.append(rec); continue
        if pur<PURITY: rec.update(label=dom,kept=False,reason="impure_window"); manifest.append(rec); continue
        rec["label"]=dom
        prri=ppg_rri(bvp[i:i+wlen])
        feats=hrv_from_rri(prri) if prri is not None else None
        if feats is None: rec.update(kept=False,reason="ppg_peak_fail"); manifest.append(rec); continue
        # ECG HRV (same window) and wrist motion
        erri=ecg_rri(ecg[int(t0*ECG_FS):int(t1*ECG_FS)])
        eh=hrv_from_rri(erri) if erri is not None else None
        mag=np.sqrt((wacc[int(t0*ACC_FS):int(t1*ACC_FS)]**2).sum(axis=1))
        motion=float(np.std(mag)) if len(mag)>0 else np.nan
        feats.update(label=LABEL_MAP[dom], subject=subj,
                     ecg_RMSSD=(eh["RMSSD"] if eh else np.nan),
                     ecg_SDNN=(eh["SDNN"] if eh else np.nan),
                     ecg_meanHR=(eh["meanHR"] if eh else np.nan), motion=motion)
        rows.append(feats)
        rec.update(kept=True,reason="kept"); manifest.append(rec)

df=pd.DataFrame(rows).dropna(subset=PPG_FEATS).reset_index(drop=True)
mdf=pd.DataFrame(manifest)
df.to_csv(f"{OUT}/features_v2.csv",index=False); mdf.to_csv(f"{OUT}/window_manifest.csv",index=False)
print("Kept feature windows:", df.shape, "| manifest rows:", mdf.shape)
print("Label balance:", df["label"].value_counts().to_dict())

In [ ]:
# 6. Manifest / rejection summary (incl. whether rejection hits stress windows harder)
rej = mdf[mdf.reason!="kept"]
print("Rejections by reason:", mdf.reason.value_counts().to_dict())
# among windows that were baseline(1) or stress(2), ppg_peak_fail rate per condition
cand = mdf[mdf.label.isin([1,2])].copy()
cand["is_stress"]=(cand.label==2)
fail = cand.groupby("is_stress").apply(lambda g:(g.reason=="ppg_peak_fail").mean())
print("PPG peak-failure rate - non-stress vs stress:", {bool(k):round(float(v),3) for k,v in fail.items()})
kept = cand.groupby("is_stress").apply(lambda g:(g.reason=="kept").mean())
print("Window retention - non-stress vs stress:", {bool(k):round(float(v),3) for k,v in kept.items()})

In [ ]:
# 7. LOSO evaluation: majority baseline + feature subsets, with extra metrics + saved predictions
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, recall_score

def loso(df, cols, mk, save_name=None):
    accs=[];f1s=[];aucs=[];srec=[];spec=[];preds=[]
    for ts in SUBJECTS:
        tr=df[df.subject!=ts]; te=df[df.subject==ts]
        if te.empty or tr.label.nunique()<2: continue
        if mk=="majority":
            maj=int(tr.label.mode()[0]); pred=np.full(len(te),maj); proba=None
        else:
            sc=StandardScaler().fit(tr[cols]); m=mk().fit(sc.transform(tr[cols]),tr.label)
            Xte=sc.transform(te[cols]); pred=m.predict(Xte)
            proba=m.predict_proba(Xte)[:,1] if hasattr(m,"predict_proba") else None
        accs.append(accuracy_score(te.label,pred))
        f1s.append(f1_score(te.label,pred,average="macro",zero_division=0))
        srec.append(recall_score(te.label,pred,pos_label=1,zero_division=0))
        spec.append(recall_score(te.label,pred,pos_label=0,zero_division=0))
        if proba is not None and te.label.nunique()==2: aucs.append(roc_auc_score(te.label,proba))
        if save_name is not None:
            for yt,yp in zip(te.label.values,pred): preds.append((ts,int(yt),int(yp)))
    if save_name is not None:
        pd.DataFrame(preds,columns=["subject","y_true","y_pred"]).to_csv(f"{OUT}/pred_{save_name}.csv",index=False)
    return dict(acc=np.mean(accs),f1=np.mean(f1s),auc=(np.mean(aucs) if aucs else np.nan),
                stress_recall=np.mean(srec),nonstress_spec=np.mean(spec),
                f1_std=np.std(f1s),n=len(f1s))

LR=lambda:LogisticRegression(max_iter=1000)
RF=lambda:RandomForestClassifier(n_estimators=300,random_state=42)
configs=[("Majority baseline","majority",None),
         ("mean-HR only",LR,["meanHR"]),
         ("Variability only (no meanHR)",LR,VARIABILITY_FEATS),
         ("Time-domain (incl meanHR)",LR,TIME_FEATS),
         ("Frequency-domain only",LR,FREQ_FEATS),
         ("All features (LR)",LR,PPG_FEATS),
         ("All features (RF)",RF,PPG_FEATS)]
res=[]
for name,mk,cols in configs:
    r=loso(df,cols,mk,save_name=name.replace(" ","_").replace("(","").replace(")",""))
    r["model"]=name; res.append(r)
results=pd.DataFrame(res)[["model","acc","f1","auc","stress_recall","nonstress_spec","f1_std","n"]].round(3)
results.to_csv(f"{OUT}/results_v2.csv",index=False)
print(results.to_string(index=False))

In [ ]:
# 8. Signal validity: PPG-derived vs ECG-derived HRV (agreement, not just correlation)
val=df.dropna(subset=["ecg_RMSSD","ecg_SDNN"]).copy()
def agreement(ppg,ecg):
    ppg=np.asarray(ppg,float); ecg=np.asarray(ecg,float)
    diff=ppg-ecg; r=spstats.pearsonr(ppg,ecg)[0]
    return dict(n=len(ppg),pearson_r=round(r,3),mae=round(np.mean(np.abs(diff)),2),
                bias=round(np.mean(diff),2), loa_lo=round(np.mean(diff)-1.96*np.std(diff),2),
                loa_hi=round(np.mean(diff)+1.96*np.std(diff),2))
print("RMSSD  PPG-vs-ECG:", agreement(val.RMSSD, val.ecg_RMSSD))
print("SDNN   PPG-vs-ECG:", agreement(val.SDNN,  val.ecg_SDNN))
print("meanHR PPG-vs-ECG:", agreement(val.meanHR, val.ecg_meanHR))

In [ ]:
# 9. Does motion (wrist ACC) explain results? Split low/high motion and re-evaluate
thr=df.motion.median()
lo=df[df.motion<=thr]; hi=df[df.motion>thr]
print(f"Motion split at std={thr:.1f}: low-motion n={len(lo)}, high-motion n={len(hi)}")
for name,sub in [("low-motion",lo),("high-motion",hi)]:
    if sub.subject.nunique()<3: print(name,"too few subjects"); continue
    r=loso(sub,PPG_FEATS,RF)
    print(f"{name}: RF macro-F1={r['f1']:.3f}, AUROC={r['auc']:.3f}, n_folds={r['n']}")
    v=sub.dropna(subset=["ecg_RMSSD"])
    if len(v)>10:
        a=agreement(v.RMSSD,v.ecg_RMSSD); print(f"   RMSSD PPG-vs-ECG MAE={a['mae']} ms, r={a['pearson_r']}")

In [ ]:
# 10. Summary block to paste into the paper
L=["## Extended results (v2)",""]
L.append("### LOSO (baseline vs stress), Logistic Regression headline")
L.append("| Model | Acc | Macro-F1 | AUROC | StressRecall | NonStressSpec | F1 std |")
L.append("|---|---|---|---|---|---|---|")
for _,r in results.iterrows():
    L.append(f"| {r['model']} | {r['acc']:.3f} | {r['f1']:.3f} | {r['auc']:.3f} | {r['stress_recall']:.3f} | {r['nonstress_spec']:.3f} | {r['f1_std']:.3f} |")
L+=["","### PPG vs ECG agreement (kept windows)"]
L.append(f"- RMSSD: {agreement(val.RMSSD,val.ecg_RMSSD)}")
L.append(f"- SDNN: {agreement(val.SDNN,val.ecg_SDNN)}")
L+=["","### Window counts / retention"]
L.append(f"- kept windows: {len(df)}  (non-stress {int((df.label==0).sum())}, stress {int((df.label==1).sum())})")
L.append(f"- rejections: {mdf.reason.value_counts().to_dict()}")
summary="\n".join(L)
open(f"{OUT}/summary_v2.md","w").write(summary)
print(summary)

In [ ]:
# 11. Bland-Altman figure: PPG vs ECG RMSSD
import matplotlib.pyplot as plt
m=(val.RMSSD+val.ecg_RMSSD)/2; d=val.RMSSD-val.ecg_RMSSD
fig,ax=plt.subplots(figsize=(5,4))
ax.scatter(m,d,alpha=0.3,s=10)
ax.axhline(d.mean(),c="k"); ax.axhline(d.mean()+1.96*d.std(),c="gray",ls="--"); ax.axhline(d.mean()-1.96*d.std(),c="gray",ls="--")
ax.set_xlabel("Mean of PPG & ECG RMSSD (ms)"); ax.set_ylabel("PPG - ECG RMSSD (ms)")
ax.set_title("Bland-Altman: PPG vs ECG RMSSD")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_blandaltman.png",dpi=200); plt.show()
print("saved outputs to", OUT, ":", sorted(os.listdir(OUT)))

## Next steps
- Download `outputs_v2/` (results_v2.csv, window_manifest.csv, pred_*.csv, summary_v2.md, fig_blandaltman.png).
- Paste `summary_v2.md` numbers into the paper's new ablation table + signal-validity section.
